#### verificar si los 32 participantes vieron los mismos 40 Experiment_id

In [1]:
from __future__ import annotations
from pathlib import Path
import pandas as pd

RATINGS_PATH: Path = Path("../dataset/raw/metadata/participant_ratings.xls")

In [4]:
ratings_df: pd.DataFrame = pd.read_excel(RATINGS_PATH)
ratings_df.head()

,Participant_id,Trial,Experiment_id,Start_time,Valence,Arousal,Dominance,Liking,Familiarity
0,1,1,5,1695918,6.96,3.92,7.19,6.05,4.0
1,1,2,18,2714905,7.23,7.15,6.94,8.01,4.0
2,1,3,4,3586768,4.94,6.01,6.12,8.06,4.0
3,1,4,24,4493800,7.04,7.09,8.01,8.22,4.0
4,1,5,20,5362005,8.26,7.91,7.19,8.13,1.0


In [5]:
print("Shape:", ratings_df.shape)
print("Columns:", ratings_df.columns.tolist())

ratings_df.info()


Shape: (1280, 9)
Columns: ['Participant_id', 'Trial', 'Experiment_id', 'Start_time', 'Valence', 'Arousal', 'Dominance', 'Liking', 'Familiarity']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1280 entries, 0 to 1279
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Participant_id  1280 non-null   int64  
 1   Trial           1280 non-null   int64  
 2   Experiment_id   1280 non-null   int64  
 3   Start_time      1280 non-null   int64  
 4   Valence         1280 non-null   float64
 5   Arousal         1280 non-null   float64
 6   Dominance       1280 non-null   float64
 7   Liking          1280 non-null   float64
 8   Familiarity     1160 non-null   float64
dtypes: float64(5), int64(4)
memory usage: 90.1 KB


In [6]:
trials_per_participant: pd.DataFrame = (
    ratings_df
    .groupby("Participant_id")["Trial"]
    .nunique()
    .reset_index(name="num_trials")
)

trials_per_participant

,Participant_id,num_trials
0,1,40
1,2,40
2,3,40
3,4,40
4,5,40
5,6,40
6,7,40
7,8,40
8,9,40
9,10,40


In [7]:
experiments_per_participant: pd.DataFrame = (
    ratings_df
    .groupby("Participant_id")["Experiment_id"]
    .nunique()
    .reset_index(name="num_experiments")
)

experiments_per_participant

,Participant_id,num_experiments
0,1,40
1,2,40
2,3,40
3,4,40
4,5,40
5,6,40
6,7,40
7,8,40
8,9,40
9,10,40


In [8]:
global_experiments: set[int] = set(
    ratings_df["Experiment_id"].astype(int).unique()
)

print("Cantidad global:", len(global_experiments))
print("Experiment_id globales:", sorted(global_experiments))

Cantidad global: 40
Experiment_id globales: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40)]


In [9]:
comparison_rows: list[dict[str, object]] = []

for participant_id, participant_df in ratings_df.groupby("Participant_id"):
    participant_experiments: set[int] = set(
        participant_df["Experiment_id"].astype(int).unique()
    )

    missing_experiments: set[int] = (
        global_experiments - participant_experiments
    )

    extra_experiments: set[int] = (
        participant_experiments - global_experiments
    )

    comparison_rows.append(
        {
            "Participant_id": int(participant_id),
            "num_experiments": len(participant_experiments),
            "same_as_global": participant_experiments == global_experiments,
            "missing_experiments": sorted(missing_experiments),
            "extra_experiments": sorted(extra_experiments),
        }
    )

comparison_df: pd.DataFrame = pd.DataFrame(comparison_rows)

comparison_df

,Participant_id,num_experiments,same_as_global,missing_experiments,extra_experiments
0,1,40,True,[],[]
1,2,40,True,[],[]
2,3,40,True,[],[]
3,4,40,True,[],[]
4,5,40,True,[],[]
5,6,40,True,[],[]
6,7,40,True,[],[]
7,8,40,True,[],[]
8,9,40,True,[],[]
9,10,40,True,[],[]


In [10]:
all_same: bool = bool(comparison_df["same_as_global"].all())

if all_same:
    print("✅ Todos los participantes vieron los mismos 40 Experiment_id.")
else:
    print("⚠️ Hay diferencias entre participantes.")
    display(comparison_df[comparison_df["same_as_global"] == False])

✅ Todos los participantes vieron los mismos 40 Experiment_id.


In [11]:
presentation_order_df: pd.DataFrame = (
    ratings_df
    .sort_values(["Participant_id", "Trial"])
    [["Participant_id", "Trial", "Experiment_id"]]
)

presentation_order_df.head(80)

,Participant_id,Trial,Experiment_id
0,1,1,5
1,1,2,18
2,1,3,4
3,1,4,24
4,1,5,20
...,...,...,...
75,2,36,36
76,2,37,3
77,2,38,34
78,2,39,10


In [12]:
pivot_trials: pd.DataFrame = ratings_df.pivot(
    index="Trial",
    columns="Participant_id",
    values="Experiment_id",
)

pivot_trials

Participant_id,1,2,3,4,5,6,7,8,9,10,...,23,24,25,26,27,28,29,30,31,32
Trial,,,,,,,,,,,,,,,,,,,,,
1,5,27,16,20,40,31,17,22,1,11,...,6,19,24,40,11,12,6,24,19,28
2,18,17,6,30,12,5,27,34,9,24,...,22,17,28,11,33,34,25,25,13,32
3,4,35,25,11,37,38,30,9,7,8,...,9,23,30,10,38,2,8,13,34,14
4,24,31,22,12,26,20,29,32,10,32,...,5,20,38,27,22,7,5,37,2,13
5,20,38,32,10,25,32,3,30,14,37,...,18,22,9,9,7,21,18,14,27,9
6,31,32,18,7,19,11,4,11,35,14,...,35,27,17,32,6,22,37,39,11,15
7,40,30,1,9,11,18,26,7,4,7,...,25,18,6,36,14,32,3,1,28,33
8,39,33,26,6,2,7,34,23,24,12,...,20,30,23,5,32,33,12,20,26,2
9,13,24,29,39,32,15,33,4,18,19,...,10,6,4,29,17,15,39,23,21,8
